# RAG PIPELINE- ## data ingestion to vector db pipeline


In [1]:
import os
# os -> File aur folder paths manage karne ke liye

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
# PDF files se text extract karne ke liye
# Dono loaders alternatives hain; requirement ke according ek use karo

from langchain_text_splitters import RecursiveCharacterTextSplitter
# Large text ko smaller chunks mein divide karne ke liye (Chunking)

from pathlib import Path
# File aur folder paths ko easily handle karne ke liye

C:\Users\RITIKA JAISWAL\AppData\Local\Temp\ipykernel_3472\4144427834.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
#Multiple PDFs ko automatically load karke ek common document list banana, jise hum next step mein chunking ke liye use karenge
def process_all_pdfs(pdf_directory):

    """Process all PDF files inside the directory"""

    all_documents = []  # Sabhi PDFs ke documents store karne ke liye

    pdf_dir = Path(pdf_directory)  # Folder path ko Path object mein convert kar rhe hain

    # Find all PDF files, including subfolders
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    # Total kitni PDF files mili hain
    print(f"Found {len(pdf_files)} PDF files to process")

    # Ek-ek PDF file ko process karenge
    for pdf_file in pdf_files:

        print(f"\nProcessing: {pdf_file.name}")

        try:
            # PDFLoader object create karke PDF load kar rhe hain
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Har document ke metadata mein source information add kar rhe hain
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            # Current PDF ke documents ko main list mein add kar rhe hain
            all_documents.extend(documents)

            print(f"✓ Loaded {len(documents)} pages")

        except Exception as e:
            # Agar kisi PDF ko load karne mein error aaye toh handle karenge
            print(f"✗ Error: {e}")

    # Total kitne documents/pages load hue
    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents  # Complete document list return kar rhe hain


# Process all PDFs inside data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: 01_python_programming.pdf
✓ Loaded 1 pages

Processing: 02_machine_learning.pdf
✓ Loaded 1 pages

Processing: 03_rag_and_langchain.pdf
✓ Loaded 1 pages

Processing: 04_cyber_security.pdf
✓ Loaded 1 pages

Total documents loaded: 4


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-31T06:36:29+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-31T06:36:29+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_python_programming.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_python_programming.pdf', 'file_type': 'pdf'}, page_content='Python Programming Introduction\nLearning notes for document-loading and RAG practice\nWhat is Python?\nPython is a high-level, interpreted programming language. It is known for readable syntax and a large\necosystem of libraries. Python was created by Guido van Rossum and first released in 1991.\nKey Features\nPython is easy to learn, supports multiple programming styles, runs on different operating systems, and\nhas a large standard library. It also has a strong developer community.\nCommon Data Types\nCommon bui

In [4]:


# Text splitting: Large documents ko smaller chunks mein divide karna
# Chunking RAG mein relevant information ko easily retrieve karne mein help karti hai.

def split_documents(documents, chunk_size=1000, chunk_overlap=200):

    # RecursiveCharacterTextSplitter object create kar rhe hain
    text_splitter = RecursiveCharacterTextSplitter(

        chunk_size=chunk_size,       
        # Har chunk mein maximum 1000 characters rakhne ki koshish karega.

        chunk_overlap=chunk_overlap, 
        # Consecutive chunks mein 200 characters common rahenge,
        # taaki previous aur next chunk ka context maintain rahe.

        length_function=len,         
        # Text ki length characters mein calculate karega.

        separators=["\n\n", "\n", " ", ""]
        # Split karne ki priority:
        # 1. \n\n -> Paragraph ke basis par
        # 2. \n   -> New line ke basis par
        # 3. " "  -> Words ke basis par
        # 4. ""   -> Individual characters ke basis par
    )

    # Original documents ko smaller chunks mein divide kar rhe hain
    split_docs = text_splitter.split_documents(documents)

    # Original documents aur generated chunks ka count show karega
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # First chunk ka example display kar rhe hain
    if split_docs:

        print("\nExample chunk:")

        # First chunk ka actual text (starting 200 characters)
        print(f"Content: {split_docs[0].page_content[:200]}...")

        # First chunk ki metadata (source file, page number etc.)
        print(f"Metadata: {split_docs[0].metadata}")

    # Saare generated chunks function ke bahar return kar rhe hain
    return split_docs


# All PDF documents ko chunks mein convert kar rhe hain
chunks = split_documents(all_pdf_documents)

# Total generated chunks count karna
len(chunks)

Split 4 documents into 8 chunks

Example chunk:
Content: Python Programming Introduction
Learning notes for document-loading and RAG practice
What is Python?
Python is a high-level, interpreted programming language. It is known for readable syntax and a lar...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-31T06:36:29+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-31T06:36:29+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\01_python_programming.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '01_python_programming.pdf', 'file_type': 'pdf'}


8